In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = 2023 - df["year"]
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [13]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [14]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)

In [15]:
import os, joblib
import lightgbm as lgb

os.makedirs("../models", exist_ok=True)

def train_lgbm(name, **extra):
    path = f"../models/{name}.joblib"
    if os.path.exists(path):
        print(f"Loaded {name}")
        return joblib.load(path)
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                          min_child_samples=20, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=42, verbose=-1, **extra)
    m.fit(train_fit[features], np.log(train_fit["price"]),
          eval_set=[(valid[features], np.log(valid["price"]))],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    joblib.dump(m, path)
    return m

mid  = train_lgbm("lgbm_mid")
low  = train_lgbm("lgbm_q10", objective="quantile", alpha=0.10)
high = train_lgbm("lgbm_q90", objective="quantile", alpha=0.90)

Loaded lgbm_mid
Loaded lgbm_q10
Loaded lgbm_q90


In [16]:
yv = np.log(valid["price"])
lo_v, hi_v = low.predict(valid[features]), high.predict(valid[features])
scores = np.maximum(lo_v - yv, yv - hi_v)
q = np.quantile(scores, 0.80)

In [17]:
X = test[features]
test = test.assign(
    pred_mid=np.exp(mid.predict(X)),
    pred_low=np.exp(low.predict(X) - q),
    pred_high=np.exp(high.predict(X) + q),
)
# Make sure low <= mid <= high
test["pred_low"] = test[["pred_low", "pred_mid"]].min(axis=1)
test["pred_high"] = test[["pred_high", "pred_mid"]].max(axis=1)

In [18]:
inside = test["price"].between(test["pred_low"], test["pred_high"]).mean()
below = (test["price"] < test["pred_low"]).mean()
above = (test["price"] > test["pred_high"]).mean()
print(f"Inside range: {inside:.1%}  (target ~80%)")
print(f"Below range:  {below:.1%}  (target ~10%)")
print(f"Above range:  {above:.1%}  (target ~10%)")

test["range_width_pct"] = (test["pred_high"] - test["pred_low"]) / test["pred_mid"]
bands = pd.cut(test["price"], [0, 10_000, 20_000, 35_000, 60_000, 200_000])
test.groupby(bands, observed=True)["range_width_pct"].median().round(3)

Inside range: 80.0%  (target ~80%)
Below range:  9.8%  (target ~10%)
Above range:  10.2%  (target ~10%)


price
(0, 10000]         0.467
(10000, 20000]     0.241
(20000, 35000]     0.177
(35000, 60000]     0.158
(60000, 200000]    0.183
Name: range_width_pct, dtype: float64

In [19]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

14,740 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17621.44,19523.99,21524.19,0.75
221479,Ford,E350 Super Duty E-350 SD,2006,96633.0,Yes,6999.0,10859.24,25302.53,32026.77,0.72
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,12741.51,31996.30,60223.82,0.70
96874,Chevrolet,Suburban 1500 LT,2007,142672.0,Yes,2999.0,7828.36,9743.78,12129.63,0.69
124379,Chevrolet,Express 2500,1999,81711.0,No,4995.0,9824.97,15788.95,23363.43,0.68
628409,Subaru,Impreza WRX,2009,163425.0,No,4899.0,10816.03,14353.07,14760.84,0.66
252260,Ford,Transit-150 T-150,2020,33199.0,Yes,13795.0,22248.22,39444.26,53037.40,0.65
593359,Nissan,Rogue Sport S,2019,55241.0,Yes,6950.0,16888.69,19269.23,21706.76,0.64
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,5763.01,8209.89,20931.54,0.63
186591,Ford,Custom Sedan,1949,228.0,No,21000.0,25022.34,56299.54,139475.77,0.63


In [20]:
df["price"].value_counts().head(15)
df.loc[df["price"] == 2500, "seller_name"].value_counts().head(10)

seller_name
Car Collection Inc                   7
Suburban Ford of Sterling Heights    2
Corwin Public Wholesale              2
REM Motors LLC                       2
Central FL Motors LLC                2
Shea Buick GMC                       1
Mike Molstead Motors                 1
Starfire Auto Inc.                   1
Putnam Auto Sales Inc                1
A2Z Auto Group                       1
Name: count, dtype: int64

In [21]:
def classify(row):
    if row["year"] < 2000:
        return "Classic (out of scope)"
    if row["discount"] > 0.5:
        return "Likely data error"
    if row["accidents_or_damage"] == "Yes":
        return "Accident history"
    return "Possible deal"

deals["flag_type"] = deals.apply(classify, axis=1)
deals["flag_type"].value_counts()

flag_type
Possible deal             11289
Accident history           3300
Classic (out of scope)       91
Likely data error            60
Name: count, dtype: int64

In [22]:
real = deals[deals["flag_type"] == "Possible deal"]
real[cols].head(20).round(2)

,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
728157,Volkswagen,Golf Alltrack TSI S,2019,45000.0,No,11000.0,21095.03,21874.93,27880.84,0.50
580659,Nissan,Sentra S,2019,115000.0,Unknown,6000.0,10265.87,11928.26,16813.78,0.50
223741,Ford,F-350 XLT Crew Cab Super Duty,2003,279321.0,Unknown,8300.0,8606.48,16438.82,18425.25,0.50
93921,Cadillac,Escalade Premium,2013,8300.0,Unknown,29500.0,29551.77,58338.70,84654.10,0.49
30068,BMW,Z3,2001,132000.0,Unknown,5500.0,5515.06,10870.34,12731.38,0.49
591404,Nissan,Rogue Select S,2015,86000.0,Unknown,6950.0,10578.02,13699.63,16706.53,0.49
502164,Mazda,Mazda5 Sport,2009,171159.0,No,2999.0,4190.18,5885.03,8267.94,0.49
30058,BMW,Z3 2.3 Roadster,2000,120700.0,Unknown,5750.0,6831.44,11216.92,12430.34,0.49
107975,Chevrolet,Tahoe LT,2012,160282.0,No,7800.0,13142.94,15146.17,16786.23,0.49
422917,Kia,Optima EX,2009,101186.0,No,4998.0,7473.54,9679.65,12380.84,0.48
